# 02 — Feature Engineering

Create lag features, rolling window statistics, seasonal dummies, holiday flags, and exogenous features for demand forecasting.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import load_processed
from src.features import (
    create_feature_pipeline, train_val_test_split, get_feature_columns
)
from src.config import CATEGORY_COL, TARGET_COL

print('Libraries loaded.')

In [ ]:
# Load processed daily data
daily = load_processed('daily_category_demand')
print(f'Daily data: {len(daily):,} rows, {daily[CATEGORY_COL].nunique()} categories')
daily.head(3)

## Feature Engineering Pipeline

In [ ]:
featured = create_feature_pipeline(daily)
print(f'\nFeatured dataset: {featured.shape[0]:,} rows × {featured.shape[1]:,} columns')
print(f'Feature columns: {len(get_feature_columns(featured))}')

In [ ]:
# View features for one category
sample_cat = featured[featured[CATEGORY_COL] == featured[CATEGORY_COL].value_counts().index[0]]
print(f'Category: {sample_cat[CATEGORY_COL].iloc[0]}')
sample_cat.head(5).T

## Train/Validation/Test Split

In [ ]:
train, val, test = train_val_test_split(featured)

print(f'Train: {len(train):,} rows ({train["date"].min().date()} → {train["date"].max().date()})')
print(f'Val:   {len(val):,} rows ({val["date"].min().date()} → {val["date"].max().date()})')
print(f'Test:  {len(test):,} rows ({test["date"].min().date()} → {test["date"].max().date()})')

In [ ]:
# Check feature availability
features = get_feature_columns(featured)
print(f'Feature categories:')
for feat in sorted(features)[:10]:
    print(f'  ✓ {feat}')
print(f'  ... and {len(features) - 10} more')

## Correlation Analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Correlation of selected features with target
corr_features = [TARGET_COL] + [c for c in features if 'lag' in c or 'rolling' in c][:10]
corr_matrix = train[corr_features].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Feature-target correlation sorted
target_corr = train[features + [TARGET_COL]].corr()[TARGET_COL].sort_values(ascending=False)
print('Top 10 features correlated with target:')
for feat, corr in target_corr.head(11).items():
    if feat != TARGET_COL:
        print(f'  {feat:40s} {corr:+.3f}')

print(f'\nBottom 5 features (negative correlation):')
for feat, corr in target_corr.tail(5).items():
    if feat != TARGET_COL:
        print(f'  {feat:40s} {corr:+.3f}')

## Save Featured Data

In [ ]:
from src.utils import save_model
save_model(featured, 'featured_data.pkl')
print('Featured data saved.')